# 🔄 Mileage Tracking API

## iPaaS Pattern: APIM + Logic Apps + Cosmos DB

This lab demonstrates building an end-to-end integration solution using Azure's Integration Platform as a Service (iPaaS) components.

### Architecture

```
┌─────────────┐     ┌─────────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   Client    │────▶│   API Management    │────▶│    Logic App    │────▶│    Cosmos DB    │
│             │     │   • Rate Limiting   │     │   • Workflows   │     │   • NoSQL Data  │
│             │     │   • Auth & Security │     │   • Managed ID  │     │   • RBAC Auth   │
└─────────────┘     └─────────────────────┘     └─────────────────┘     └─────────────────┘
```

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed
- [An Azure Subscription](https://azure.microsoft.com/free/) with Contributor permissions
- [Sign in to Azure with Azure CLI](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id
- Adjust the location parameter according to your preferences and [product availability by region](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/)

In [3]:
import os
import sys
import json
import datetime
import hashlib

# Add shared utilities
sys.path.insert(1, '../../shared')
import utils

# Configuration
deployment_name = "mileage-tracking"
resource_group_location = "westus2"

# Get subscription ID for unique naming
subscription_id = utils.get_current_subscription()
unique_suffix = hashlib.md5(subscription_id.encode()).hexdigest()[:8] if subscription_id else "default"

# Resource names
resource_group_name = f"rg-{deployment_name}"

print(f"📋 Configuration:")
print(f"   Resource Group: {resource_group_name}")
print(f"   Location: {resource_group_location}")
print(f"   Unique Suffix: {unique_suffix}")

✅ Retrieved Azure account ⌚ 23:20:24
ℹ️  Using Subscription: ME-MngEnvMCAP198880-odaibert-1 (2557a18b-db35-40e8-9977-08845cf4192a)
📋 Configuration:
   Resource Group: rg-mileage-tracking
   Location: westus2
   Unique Suffix: beef5c0b


<a id='1'></a>
### 1️⃣ Verify Azure CLI and connected subscription

The following commands ensure that you have the Azure CLI installed and connected to your Azure subscription.

In [4]:
# Verify Azure CLI
output = utils.run("az account show", "Azure CLI is configured", "Azure CLI not configured - run 'az login'")

if output.success and output.json_data:
    print(f"\n📌 Subscription: {output.json_data['name']}")
    print(f"📌 Subscription ID: {output.json_data['id']}")
    print(f"📌 Tenant ID: {output.json_data['tenantId']}")

✅ Azure CLI is configured ⌚ 23:20:29

📌 Subscription: ME-MngEnvMCAP198880-odaibert-1
📌 Subscription ID: 2557a18b-db35-40e8-9977-08845cf4192a
📌 Tenant ID: 6f4e776f-28a2-4303-879c-d1dba3028420


<a id='2'></a>
### 2️⃣ Create Resource Group

Create the Azure resource group that will contain all the lab resources.

In [5]:
# Create resource group
utils.create_resource_group(resource_group_name, resource_group_location)

✅ Created resource group 'rg-mileage-tracking' in westus2 ⌚ 23:20:40


True

<a id='3'></a>
### 3️⃣ Deploy Infrastructure using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview) to declaratively define all resources.

The deployment creates:
- **Cosmos DB Account** with SQL API database and container
- **Logic App** with System-assigned Managed Identity
- **API Management** (BasicV2 tier) with rate-limited API
- **RBAC Role Assignment** for Logic App to access Cosmos DB

⏱️ *Note: APIM BasicV2 deployment takes approximately 5-10 minutes*

In [6]:
# Deploy infrastructure
outputs = utils.deploy_bicep(
    resource_group=resource_group_name,
    template_file="main.bicep",
    parameters={"location": resource_group_location}
)

if outputs:
    print("\n📦 Deployed Resources:")
    for key, value in outputs.items():
        print(f"   {key}: {value['value'] if isinstance(value, dict) else value}")

ℹ️  Deploying main.bicep...
✅ Deployment completed ⌚ 23:23:01

📦 Deployed Resources:
   apimGatewayUrl: https://apim-gateway-learning-ktdpltkqf77ne.azure-api.net
   apimMileageApiPath: https://apim-gateway-learning-ktdpltkqf77ne.azure-api.net/mileage/calculate
   apimName: apim-gateway-learning-ktdpltkqf77ne
   containerName: MileageData
   cosmosAccountEndpoint: https://cosmos-mileage-ktdpltkqf77ne.documents.azure.com:443/
   cosmosAccountName: cosmos-mileage-ktdpltkqf77ne
   databaseName: BackendSystems
   logicAppName: logic-mileage-orchestrator
   logicAppPrincipalId: 2bbf6cde-c678-49b7-9312-bb0b1af15dca
   logicAppUrl: https://prod-25.westus2.logic.azure.com:443/workflows/fa23ccc0010744c685bc0c794125d2a8?api-version=2019-05-01&sp=%2F%2F%2A&sv=1.0&sig=UO4SJ908nzke71BcqWnZn3DWEeSJSGIhSlIECVyy_4k
   portalDesignerUrl: https://portal.azure.com/#@/resource/subscriptions/2557a18b-db35-40e8-9977-08845cf4192a/resourceGroups/rg-mileage-tracking/providers/Microsoft.Logic/workflows/logic-mil

<a id='4'></a>
### 4️⃣ Get Deployment Outputs

Retrieve the required outputs from the Bicep deployment.

In [7]:
# Get deployment outputs
outputs = utils.get_deployment_outputs(resource_group_name)

if outputs:
    apim_gateway_url = outputs.get('apimGatewayUrl', '')
    apim_name = outputs.get('apimName', '')
    cosmos_account_name = outputs.get('cosmosAccountName', '')
    logic_app_name = outputs.get('logicAppName', '')
    
    print(f"\n🔗 API Management Gateway: {apim_gateway_url}")
    print(f"🔗 Cosmos DB Account: {cosmos_account_name}")
    print(f"🔗 Logic App: {logic_app_name}")
else:
    print("❌ Failed to retrieve deployment outputs")

✅ Retrieved deployment outputs ⌚ 23:24:47

🔗 API Management Gateway: https://apim-gateway-learning-ktdpltkqf77ne.azure-api.net
🔗 Cosmos DB Account: cosmos-mileage-ktdpltkqf77ne
🔗 Logic App: logic-mileage-orchestrator


<a id='5'></a>
### 5️⃣ Configure Logic App Cosmos DB Connector

Due to Azure Policy restrictions on Cosmos DB access keys, we need to manually configure the Logic App's Cosmos DB connector with Managed Identity authentication.

**Steps:**
1. Open the Logic App Designer in Azure Portal
2. Add the "Create or update document (V3)" action
3. Select "Logic Apps Managed Identity" for authentication
4. Configure the database and collection IDs

<div style="display: flex; gap: 10px;">
  <img src="workflow01.png" alt="Logic App Workflow Configuration" >
  <img src="workflow02.png" alt="Logic App Workflow Configuration" >
</div>

In [ ]:
# Generate Logic App Designer URL
if outputs:
    designer_url = f"https://portal.azure.com/#@/resource/subscriptions/{subscription_id}/resourceGroups/{resource_group_name}/providers/Microsoft.Logic/workflows/{logic_app_name}/designer"
    
    print("📝 Configure the Cosmos DB connector in the Logic App Designer:")
    print(f"\n🔗 {designer_url}")
    print("\n📋 Configuration values:")
    print(f"   • Cosmos DB Account: {cosmos_account_name}")
    print(f"   • Database ID: BackendSystems")
    print(f"   • Collection ID: MileageData")
    print(f"   • Authentication: Logic Apps Managed Identity")

<a id='6'></a>
### 6️⃣ Get APIM Subscription Key

Retrieve the subscription key needed to authenticate API requests.

In [ ]:
# Get APIM subscription key
result = utils.run(
    f'az rest --method post --uri "https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{resource_group_name}/providers/Microsoft.ApiManagement/service/{apim_name}/subscriptions/master/listSecrets?api-version=2023-05-01-preview" --query primaryKey -o tsv',
    "Retrieved APIM subscription key",
    "Failed to get APIM subscription key"
)

if result.success:
    subscription_key = result.output
    print(f"\n🔑 Subscription Key: {subscription_key}")

<a id='test'></a>
### 🧪 Test the Mileage API

Send a test request to the Mileage API through API Management.

In [ ]:
import requests
import uuid

# Generate unique test ID
test_id = f"mileage-{uuid.uuid4().hex[:8]}"

# Test payload
payload = {
    "id": test_id,
    "vehicle": "sedan-xyz",
    "mileage": 7500,
    "date": datetime.datetime.now().strftime("%Y-%m-%d")
}

# API endpoint
api_url = f"{apim_gateway_url}/mileage/calculate"

# Headers
headers = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": subscription_key
}

print(f"📤 Sending request to: {api_url}")
print(f"📦 Payload: {json.dumps(payload, indent=2)}")

# Make request
response = requests.post(api_url, json=payload, headers=headers)

print(f"\n📥 Response Status: {response.status_code}")
print(f"📥 Response Body: {json.dumps(response.json(), indent=2)}")

<a id='ratelimit'></a>
### 🧪 Test Rate Limiting

Send multiple requests to verify the rate limiting policy (5 calls per 60 seconds).

In [ ]:
import time

print("🔄 Testing rate limiting (sending 7 requests)...\n")

for i in range(7):
    test_payload = {
        "id": f"rate-test-{i+1}",
        "vehicle": "test-car",
        "mileage": 1000 * (i + 1),
        "date": datetime.datetime.now().strftime("%Y-%m-%d")
    }
    
    response = requests.post(api_url, json=test_payload, headers=headers)
    
    status_icon = "✅" if response.status_code == 200 else "❌"
    print(f"   Request {i+1}: {status_icon} Status {response.status_code}")
    
    if response.status_code == 429:
        print(f"      └── Rate limit exceeded! (as expected after 5 requests)")
    
    time.sleep(0.5)  # Small delay between requests

print("\n✅ Rate limiting test complete!")

<a id='cosmos'></a>
### 🧪 Verify Data in Cosmos DB

Query Cosmos DB to verify the documents were created.

In [ ]:
# Open Cosmos DB Data Explorer
data_explorer_url = f"https://portal.azure.com/#@/resource/subscriptions/{subscription_id}/resourceGroups/{resource_group_name}/providers/Microsoft.DocumentDB/databaseAccounts/{cosmos_account_name}/dataExplorer"

print("📊 View your data in Cosmos DB Data Explorer:")
print(f"\n🔗 {data_explorer_url}")
print("\n📋 Navigate to: BackendSystems > MileageData > Items")

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges.

Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.

In [ ]:
# Uncomment to delete resources
# utils.delete_resource_group(resource_group_name)
print(f"💡 To delete resources, run: az group delete --name {resource_group_name} --yes")